# 2D example
This notebook shows a simple analysis pipeline for running and analyzing a 3D simulation with DISCO-DJ.

### Imports

In [1]:
# Import modules
import os
from matplotlib import pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set_style("ticks")
import jax
from jax import config
import numpy as np
from discodj import DiscoDJ
plt.rcParams['image.cmap'] = "rocket"
print(jax.__version__)

0.7.2


In [ ]:
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95" # maximize available memory (might need to tune this in case you're running out of memory)

### Simulation settings
DISCO-DJ can be run on a GPU (*much* faster) or on a CPU. The following cell detects if a GPU is available and sets the device accordingly.

In [ ]:
# Detect if a GPU is available
devices = jax.devices()
device = "gpu" if np.any([d.platform == "gpu" for d in devices]) else "cpu"
print(device)

Now, we set the name of the analysis and the spatial dimension. The precision can be set to "single" (float32) or "double" (float64; in that case, the Jax config needs to be updated). DISCO-DJ is written in an class-based (yet functional) way, so you need to define a DiscoDJ object that will be used to perform all computations.

In [ ]:
# Set the parameters
name = "2D_analysis"
dim = 2

precision = "double"
if precision == "double":
    config.update("jax_enable_x64", True)

# The cosmology can be provided as a dictionary or a string for a pre-defined cosmology (e.g. "CamelsCV" for the Camels Cosmic Variance suite).
# cosmo = "CamelsCV"
cosmo = dict(Omega_c=0.8,  # Einstein-de Sitter Universe with Omega_m = 1 (20% of which are assumed to be baryonic)
             Omega_b=0.2,
             Omega_k=0.0,
             h=0.6774,
             n_s=-1.9,
             sigma8=0.8,
             w0=-1.0,
             wa=0.0)

# Define the boxsize and resolution
boxsize = 1.0
res = 1024

# Define DISCO-DJ object
dj = DiscoDJ(dim=dim, res=res, name=name, device=device, precision=precision, boxsize=boxsize, cosmo=cosmo)
print(dj)

Next, we compute the timetable for the cosmological growth functions.

In [ ]:
# Compute the linear power spectrum
dj = dj.with_timetables()
dj = dj.with_linear_ps(transfer_function="none") # "Eisenstein-Hu"

In [ ]:
cosmo = dj.cosmo
print(cosmo)


### Initial conditions

In [ ]:
dj = dj.with_ics(seed=0, fix_std=0.9)

k_pre, Pk_pre, _ = dj.evaluate_power_spectrum(dj.delta_ini, compute_std=False, bins=80)
plt.figure(figsize=(6, 6))
plt.loglog(k_pre, Pk_pre)
plt.xlabel(r"$k$ [$h$/Mpc]")
plt.ylabel(r"$P(k)$ [$h^{-3}$Mpc$^3$]")

### Lagrangian perturbation theory (LPT)

In [ ]:
n_order = 9
dj = dj.with_lpt(n_order=n_order, convert_to_numpy=True)

### Particle-mesh (PM) N-body simulation


In [ ]:
a_ini = 0.0
a_end = 0.5
stepper = "bullfrog"
method = "pm"
res_pm = 2 * dj.res
time_var = "a"
antialias = 2
grad_kernel_order = 0
laplace_kernel_order = 0
n_resample = 16
deconvolve = True
n_steps = 4  # 4 time steps
nlpt_order_ics = n_order

In [ ]:
X_sim, P_sim, a_sim = dj.run_nbody(a_ini=a_ini, a_end=a_end, n_steps=n_steps, res_pm=res_pm, time_var=time_var,
                                   stepper=stepper, method=method, antialias=antialias,
                                   grad_kernel_order=grad_kernel_order, laplace_kernel_order=laplace_kernel_order,
                                   nlpt_order_ics=nlpt_order_ics, n_resample=n_resample, deconvolve=deconvolve)

### Analysis

Let's compute the power spectrum:

In [ ]:
delta_sim = dj.get_delta_from_pos(X_sim) #, res=2 * dj.res, n_resample=8

In [ ]:
k, Pk, _ = dj.evaluate_power_spectrum(delta_sim, bins=dj.res//3, deconvolve=True, worder=2)
Pk_linear = dj.evaluate_linear_ps(a_end, k)  # linear expectation
_, Pk_linear_realized, _ = dj.evaluate_power_spectrum(dj.delta_ini, bins=dj.res//3)  # linear realization

In [ ]:
plt.figure(figsize=(10, 6))
plt.loglog(k, Pk, "k-", label="Non-linear")
plt.loglog(k, Pk_linear, "b:", label="Linear (Expectation)")
plt.loglog(k, Pk_linear_realized, "r-.", label="Linear (Realized)")
plt.xlabel(r"$k$ [$h$/Mpc]")
plt.ylabel(r"$P(k)$")
plt.legend(fontsize=18)

plt.figure(figsize=(14, 14))
plt.imshow(np.log10(1.1 + delta_sim))